# ❤️ Heart Disease Risk Prediction using KNN and SVM

**Standalone Google Colab notebook** — the supplied `heart_disease_dataset.csv` is embedded inside this notebook, so you do **not** need to upload the CSV separately.

Workflow: missing-value handling → feature scaling → KNN/SVM/Logistic Regression → recall-focused hyperparameter tuning → evaluation.

> **Important:** This is an educational screening aid, not a medical diagnosis.

In [ ]:
# Install required packages in Google Colab
!pip -q install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# The complete CSV dataset is embedded in this notebook.
# Running this cell recreates the CSV file in Colab automatically.
import base64
from pathlib import Path

CSV_B64 = '''...'''
csv_file = Path('/content/heart_disease_dataset.csv')
csv_file.write_bytes(base64.b64decode(CSV_B64))

print(f'Dataset created: {csv_file}')
print(f'File size: {csv_file.stat().st_size:,} bytes')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

df = pd.read_csv('/content/heart_disease_dataset.csv')

print('Dataset shape:', df.shape)
display(df.head())

In [ ]:
print('Columns:')
print(df.columns.tolist())

print('\nMissing values:')
display(df.isna().sum().to_frame('missing'))

print('\nTarget distribution:')
display(df['heart_disease'].value_counts().to_frame('count'))

In [ ]:
TARGET = 'heart_disease'
FEATURES = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal',
    'smoking', 'diabetes', 'bmi'
]

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## Why feature scaling is important

The preprocessing pipeline uses median imputation followed by `StandardScaler`.

- **KNN** is distance-based, so features with larger numeric scales can dominate the distance calculation.
- **SVM** is sensitive to feature scale because it learns a separating margin/kernel.
- Putting preprocessing inside the pipeline prevents data leakage during cross-validation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

knn_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', KNeighborsClassifier())
])

svm_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', SVC(probability=True, random_state=42))
])

logreg_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

In [ ]:
# Hyperparameter tuning with RECALL as the primary scoring metric.
# Recall is emphasized because this is a screening-style project where
# reducing false negatives is important.

knn_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}

svm_grid = {
    'model__C': [0.1, 1, 10],
    'model__kernel': ['rbf', 'linear'],
    'model__class_weight': [None, 'balanced']
}

logreg_grid = {
    'model__C': [0.1, 1, 10],
    'model__class_weight': [None, 'balanced']
}

searches = {
    'KNN': GridSearchCV(knn_pipe, knn_grid, cv=cv, scoring='recall', n_jobs=-1, refit=True),
    'SVM': GridSearchCV(svm_pipe, svm_grid, cv=cv, scoring='recall', n_jobs=-1, refit=True),
    'Logistic Regression': GridSearchCV(logreg_pipe, logreg_grid, cv=cv, scoring='recall', n_jobs=-1, refit=True)
}

for name, search in searches.items():
    print(f'\nTuning {name}...')
    search.fit(X_train, y_train)
    print('Best parameters:', search.best_params_)
    print('Best CV recall:', round(search.best_score_, 4))

In [ ]:
def evaluate_model(name, fitted_search):
    pred = fitted_search.predict(X_test)
    proba = fitted_search.predict_proba(X_test)[:, 1]

    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, proba)
    }

    print(f'\n===== {name} =====')
    print(classification_report(y_test, pred, digits=3, zero_division=0))
    return metrics, pred, proba

rows = []
predictions = {}
probabilities = {}

for name, search in searches.items():
    metrics, pred, proba = evaluate_model(name, search)
    rows.append(metrics)
    predictions[name] = pred
    probabilities[name] = proba

results = pd.DataFrame(rows).set_index('Model').round(4)
display(results)

In [ ]:
# Confusion matrix for SVM
cm = confusion_matrix(y_test, predictions['SVM'])

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('SVM Confusion Matrix')
plt.show()

In [ ]:
# Example prediction using one test record
sample = X_test.iloc[[0]]
sample_probability = searches['SVM'].predict_proba(sample)[0, 1]
sample_prediction = int(sample_probability >= 0.50)

print('Predicted class:', sample_prediction)
print('Predicted positive probability:', round(sample_probability, 4))
print('\nThis is a machine-learning screening output, not a medical diagnosis.')

## Conclusion

This notebook demonstrates classification, missing-value handling, feature scaling, hyperparameter tuning and recall-focused evaluation for the supplied heart disease dataset.

Recall is emphasized to reduce false negatives, but precision, F1, ROC-AUC and the confusion matrix should also be reviewed. The resulting model is a screening aid and **must not be treated as a medical diagnosis**.